# SCD Type 2 history

Maintain version history per FHIR resource logical ID, tracking business attributes while preserving source version metadata separately.

In [ ]:
resource_type = "Patient"
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

allowed = {"Patient", "Encounter", "Observation", "Condition"}
if resource_type not in allowed:
    raise ValueError(f"Unsupported resource_type: {resource_type}")
silver_table = f"silver_{resource_type.lower()}"
history_table = f"history_{resource_type.lower()}"
if not spark.catalog.tableExists(silver_table):
    raise ValueError(f"Missing Silver table: {silver_table}")

silver = spark.table(silver_table)
lineage = {"source_version_id","source_last_updated","extraction_timestamp","ingestion_timestamp",
           "api_url_or_params","batch_id","raw_resource_json"}
tracked = [c for c in silver.columns if c not in lineage and c not in {"resource_id"}]
if not tracked:
    tracked = ["resource_type"]
hash_parts = [F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in tracked]
incoming = (silver.withColumn("attribute_hash", F.sha2(F.concat_ws("||", *hash_parts), 256))
            .withColumn("valid_from", F.coalesce(F.to_timestamp("source_last_updated"), F.current_timestamp()))
            .withColumn("valid_to", F.lit(None).cast("timestamp"))
            .withColumn("is_active", F.lit(True))
            .withColumn("scd_processed_at", F.current_timestamp()))

if not spark.catalog.tableExists(history_table):
    (incoming.write.format("delta").mode("overwrite").saveAsTable(history_table))
    print(f"Initialized {history_table} with {incoming.count()} current versions.")
else:
    target = DeltaTable.forName(spark, history_table)
    # Expire only active records whose tracked business attributes changed.
    (target.alias("t").merge(incoming.alias("s"),
        "t.resource_id = s.resource_id AND t.is_active = true")
     .whenMatchedUpdate(
        condition="t.attribute_hash <> s.attribute_hash",
        set={"valid_to":"s.valid_from","is_active":"false","scd_processed_at":"current_timestamp()"}
     ).execute())

    active = spark.table(history_table).filter(F.col("is_active") == True).select("resource_id","attribute_hash")
    new_versions = incoming.alias("s").join(
        active.alias("a"),
        (F.col("s.resource_id") == F.col("a.resource_id")) &
        (F.col("s.attribute_hash") == F.col("a.attribute_hash")),
        "left_anti"
    )
    if new_versions.limit(1).count() > 0:
        new_versions.write.format("delta").mode("append").saveAsTable(history_table)

active_count = spark.table(history_table).filter(F.col("is_active") == True).count()
print(f"{history_table}: {active_count} active versions; history retained.")
